<a href="https://colab.research.google.com/github/timfitz04/Business-Analytics-Dissertation/blob/main/KerryWS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

https://www.kerrygaa.ie/fixtures-results/

In this file Kerry league tables were scraped and collected then stored.

Team names were then cleaned so the dataset would  merge with gaapitchfinder dataset adding longitudal and latiudal data to the orignial data that was scraped. Teams that did not match after cleaning were mannually matched.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
os.chdir("/content/drive/MyDrive/Dissertation/Kerry")
!pwd

/content/drive/MyDrive/Dissertation/Kerry


In [ ]:
!pip install bs4

ERROR: Operation cancelled by user
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 447, in run
    conflicts = self._determine_conflicts(to_install)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 578, in _determine_conflicts
    return check_install_conflicts(to_install)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/operations/check.py", line 101, in check_install_conflicts
    package_set, _ = create_package_

In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import time
time.sleep(5)   # 1 second delay between requests

compIDs_by_year = {
    2024: {
        "Division 1": 209505,
        "Division 2": 209507,
        "Division 3": 209509,
        "Division 4": 209511,
        "Division 5 A": 209513,
        "Division 5 B": 209515,
        "Division 6 A": 209521,
        "Division 6 B": 209737,
    },
    2023: {
        "Division 1": 188343,
        "Division 2": 188345,
        "Division 3": 188347,
        "Division 4": 188349,
        "Division 5 A": 188897,
        "Division 5 B": 188899,
        "Division 6": 188905,
    },
    2022: {    "Division 1": 169395,
        "Division 2": 169397,
        "Division 3": 169399,
        "Division 4": 169401,
        "Division 5": 169409,
        "Division 6 A": 169411,
        "Division 6 B": 169413},

    2021: {
        "Division 1 A": 154467,
        "Division 1 B": 154469,
        "Division 2 A": 154471,
        "Division 2 B": 154473,
        "Division 3 A": 154475,
        "Division 3 B": 154477,
        "Division 4 A": 154479,
        "Division 4 B": 154481,
        "Division 5 A": 154483,
        "Division 5 B": 154485,
        "Division 6 A": 154777,
        "Division 6 B": 154779,
        "Division 6 C": 154781},
    2019: {
        "Division 1": 113719,
        "Division 2": 113721,
        "Division 3": 113723,
        "Division 4": 113725,
        "Division 5 A": 113727,
        "Division 5 B": 113729},
    2018: {
        "Division 1": 95203,
        "Division 2": 95205,
        "Division 3": 95207,
        "Division 4": 95209,
        "Division 5 A": 95211,
        "Division 5 B": 96169}
}

all_rows = []

for year, divisions in compIDs_by_year.items():

    for division_name, compid in divisions.items():

        url = (
            "https://www.kerrygaa.ie/fixtures-results/fixtures-results-ajax/"
            f"?countyBoardID=13&ccAC=1&compID={compid}"
            "&leagueTable=Y&resultsOnly=Y&reverseDateOrder=Y&orderTBCLast=Y"
        )

        response = requests.get(url)
        time.sleep(1)
        soup = BeautifulSoup(response.text, "html.parser")


        tables = pd.read_html(str(soup))
        if len(tables) == 0:
            print(f"No table for {division_name} {year}")
            continue

        df = tables[0]



        df.columns = (
            df.columns
            .str.strip()
            .str.lower()
            .str.replace(" ", "_")
        )

        rename_map = {
            "p": "pld",
            "team": "team",
            "w": "w",
            "d": "d",
            "l": "l",
            "f": "pf",
            "a": "pa",
            "pts": "pts"
        }
        df = df.rename(columns=rename_map)


        # Calculate points difference
        df["pd"] = df["pf"].astype(int) - df["pa"].astype(int)

        # Add division + year
        df["Division"] = division_name
        df["Year"] = year
        df["County"] = "Kerry"

        df = df[["County", "Division", "Year", "team", "pld", "w", "d", "l", "pf", "pa", "pd", "pts"]]

        all_rows.append(df)


# ---- FINAL OUTPUT ----
Kerry = pd.concat(all_rows, ignore_index=True)

print(Kerry)


/tmp/ipython-input-3738294176.py:84: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(str(soup))
/tmp/ipython-input-3738294176.py:84: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(str(soup))
/tmp/ipython-input-3738294176.py:84: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(str(soup))
/tmp/ipython-input-3738294176.py:84: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(str(soup))
/tmp/ipython-input-3738294176.py:84:

    County      Division  Year             team  pld   w  d  l   pf   pa   pd  \
0    Kerry    Division 1  2024       Dr. Crokes   11  11  0  0  227  102  125   
1    Kerry    Division 1  2024    Laune Rangers   11   8  2  1  189  153   36   
2    Kerry    Division 1  2024         Rathmore   11   8  0  3  163  145   18   
3    Kerry    Division 1  2024              Spa   11   5  1  5  159  150    9   
4    Kerry    Division 1  2024        Kilcummin   11   4  3  4  142  152  -10   
..     ...           ...   ...              ...  ...  .. .. ..  ...  ...  ...   
435  Kerry  Division 5 B  2018          Moyvane    6   4  0  2   67   68   -1   
436  Kerry  Division 5 B  2018  Sneem/Derrynane    6   2  0  4   79   89  -10   
437  Kerry  Division 5 B  2018          Tuosist    6   2  0  4   76   92  -16   
438  Kerry  Division 5 B  2018         Rathmore    6   2  0  4  102  119  -17   
439  Kerry  Division 5 B  2018           Dingle    6   1  0  5   81  111  -30   

     pts  
0     22  
1    

/tmp/ipython-input-3738294176.py:84: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(str(soup))


In [ ]:
import re
def clean_club_name(name):
    name = name.lower()

    # remove trailing numbers (Thomas Davis 2 → Thomas Davis)
    name = re.sub(r"\s*\d+$", "", name)

    # remove reserve/second team indicators e.g. "2nd", "B", "C"
    name = re.sub(r"\b(2nd|second|team|b|c)\b", "", name, flags=re.IGNORECASE)

    # strip GAA-specific suffixes and organisation labels
    name = re.sub(r"\b(gaa|clg|club|lgfa|ladies)\b", "", name, flags=re.IGNORECASE)

    # remove punctuation + multiple spaces
    name = re.sub(r"[^\w\s]", "", name)
    name = " ".join(name.split())
    return name.strip()

# after scraping into Dub dataframe:
Kerry["clean_team"] = Kerry["team"].apply(clean_club_name)

In [ ]:
manual_map = {
    "ardfert football": "ardfert",
    "desmonds": "castleisland desmonds",
    "reenard": "renard",
    "piarsaigh na dromoda": "dromid pearses",
    "sneemderrynane": "sneem",
    "st marys": "st marys cahirciveen"
}


Kerry["clean_team"] = Kerry["clean_team"].replace(manual_map)


In [ ]:
coords = pd.read_csv("gaapitchfinder_data.csv")
coords["clean_team"] = coords["Club"].apply(clean_club_name)
coords = coords[coords["County"] == "Kerry"].copy()
coords = coords.drop_duplicates(
    subset="clean_team",
    keep="first"
)
coords = coords[["clean_team", "Latitude", "Longitude"]]

#coords.head(50)

In [ ]:
Kerry = Kerry.merge(coords, on="clean_team", how="left")

In [ ]:
Kerry[Kerry["Latitude"].isna()][["team","clean_team"]].drop_duplicates()

,team,clean_team


In [ ]:
Kerry.to_csv("kerry_league_tables.csv", index=False)
